In [20]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [21]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

In [22]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(base_url=anthropic_url, api_key=anthropic_api_key)

In [23]:
TOPICS = [
    "travel", 
    "food", 
    "animals", 
    "sports", 
    "music", 
    "history", 
    "job interview", 
    "IT", 
    "AI Engineering"
]

def get_system_message(topic, word):
    return f"""
        You are an expert English lexicographer producing dictionary-style entries.

        ### Task
        Analyze the word "{word}" in the context of the topic: "{topic}".

        ### Instructions
        1. Provide a precise definition (maximum 15 words).
        2. Provide the IPA phonetic transcription.
        3. Provide exactly 3 synonyms. If fewer exist, use near-synonyms marked with (~).
        4. Write one sentence using "{word}" that fits the "{topic}" context.
        5. Output ONLY the format below — no preamble, no explanation.

        ### Output Format
        Word: {word}
        Transcription: <IPA>
        Definition: <Definition>
        Synonyms: <Synonym 1>, <Synonym 2>, <Synonym 3>
        Sentence: <Sentence>
"""

In [24]:
def stream_gpt(topic, word):
    system_message = get_system_message(topic, word)
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": word}
    ]
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [25]:
def stream_claude(topic, word):
    system_message = get_system_message(topic, word)
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": word}
    ]
    stream = anthropic.chat.completions.create(
        model="claude-haiku-4-5-20251001",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [26]:
def stream_model(model, topic, word):
    if model == "GPT":
        result = stream_gpt(topic, word)
    elif model == "Claude":
        result = stream_claude(topic, word)
    else:
        raise ValueError("Unknown model")
    yield from result

In [28]:
message_input = gr.Textbox(label="Target Word:", info="Enter a word to define", lines=2)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
topic_selector = gr.Dropdown(TOPICS, label="Select Topic Context", value="IT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLM Dictionary", 
    inputs=[model_selector, topic_selector, message_input],
    outputs=[message_output], 
    examples=[
        ["GPT", "AI Engineering", "persistence"],
        ["Claude", "job interview", "aspire"]
    ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
